# Aster — GPU Worker (Kaggle 2× T4)

This notebook turns a Kaggle **GPU T4 ×2** session into Aster's AI worker.

| GPU | Runs | Used for |
|---|---|---|
| **T4 #0** | Ollama → `qwen3-vl:8b` (Q4) | Agent brain: chat, tool calling, eligibility research reasoning, document understanding, **screen-share guidance (vision)** |
| **T4 #1** | Gateway → IndicConformer ASR + EasyOCR | Hindi/Marathi speech-to-text, OCR with bounding boxes + confidence |

Everything is served behind **one authenticated gateway** (FastAPI on port 8000):

- `POST /v1/chat/completions` — OpenAI-compatible (streaming, tools, images) → Ollama
- `POST /asr` — audio + `lang` → transcript
- `POST /ocr` — image → numbered lines with bbox + confidence
- `GET /health`, `POST /warmup`

### Do we need Cloudflare? **Yes.**
Kaggle has no inbound ports. The phone backend must call this GPU in real time (voice chat, screen guidance), so this notebook opens a **Cloudflare Quick Tunnel to the gateway** (never to raw Ollama). Every request needs `Authorization: Bearer <GATEWAY_TOKEN>`. The tunnel URL changes every session, so the notebook **registers the new URL in Supabase**; the backend always reads the latest one.

### Before running
1. **Settings → Accelerator → GPU T4 ×2**, **Internet → On**.
2. **Add-ons → Secrets**, add:
   - `GATEWAY_TOKEN` — long random string (same value in the backend's env)
   - `HF_TOKEN` — Hugging Face token; accept the IndicConformer model terms on its HF page first
   - `SUPABASE_URL`, `SUPABASE_ANON_KEY`, `GPU_REGISTER_SECRET` — optional, for auto-registration
3. For demo day: **Save Version → Save & Run All** runs headless for up to ~12 h; the last cell keeps it alive.

### One-time Supabase setup
If you applied `supabase/migrations/0001_init.sql` (recommended), these tables and the function already exist; only run the `insert into gpu_secrets` line with your real secret. Otherwise run all of this in the Supabase SQL editor:
```sql
create table if not exists gpu_endpoints (
  name text primary key,
  url text not null,
  models jsonb,
  last_seen timestamptz default now()
);
create table if not exists gpu_secrets (id int primary key default 1, secret text not null);
alter table gpu_endpoints enable row level security;   -- no policies: only service role (backend) reads
alter table gpu_secrets  enable row level security;   -- nobody reads this via API

insert into gpu_secrets (id, secret) values (1, 'PUT-THE-SAME-GPU_REGISTER_SECRET-HERE')
on conflict (id) do update set secret = excluded.secret;

create or replace function register_gpu(p_secret text, p_name text, p_url text, p_models jsonb)
returns void language plpgsql security definer set search_path = public as $$
begin
  if p_secret is distinct from (select secret from gpu_secrets where id = 1) then
    raise exception 'unauthorized';
  end if;
  insert into gpu_endpoints (name, url, models, last_seen)
  values (p_name, p_url, p_models, now())
  on conflict (name) do update set url = excluded.url, models = excluded.models, last_seen = now();
end $$;
grant execute on function register_gpu(text, text, text, jsonb) to anon;
```
Backend: `select url from gpu_endpoints where name = 'kaggle-main' and last_seen > now() - interval '3 minutes'` → if empty, fall back to Sarvam/Bhashini/hosted LLM.

## 1. Config

In [ ]:
import os

# "fast"  (default): qwen3-vl:8b on GPU0, ASR + OCR on GPU1
# "smart": bigger brain split across both GPUs; local ASR off (backend uses Sarvam/Bhashini), OCR on CPU
MODE = "fast"

BRAIN_MODEL = "qwen3-vl:8b" if MODE == "fast" else "qwen3-vl:30b"   # confirm tag exists in your Ollama version
NUM_CTX = 16384        # per request; agent turns + one screenshot fit comfortably
NUM_PARALLEL = 3       # concurrent chats on the brain (KV cache = NUM_CTX × NUM_PARALLEL)
WORKER_NAME = "kaggle-main"
GATEWAY_PORT = 8000

def secret(name, required=False):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        if required:
            raise RuntimeError(f"Missing Kaggle secret: {name}")
        return None

GATEWAY_TOKEN = secret("GATEWAY_TOKEN", required=True)
HF_TOKEN = secret("HF_TOKEN")
SUPABASE_URL = secret("SUPABASE_URL")
SUPABASE_ANON_KEY = secret("SUPABASE_ANON_KEY")
GPU_REGISTER_SECRET = secret("GPU_REGISTER_SECRET")

OLLAMA_GPUS  = "0" if MODE == "fast" else "0,1"
GATEWAY_GPUS = "1" if MODE == "fast" else ""
print("Mode:", MODE, "| Brain:", BRAIN_MODEL, "| Ollama GPUs:", OLLAMA_GPUS, "| Gateway GPUs:", GATEWAY_GPUS or "CPU")

## 2. Check GPUs (expect two Tesla T4)

In [ ]:
!nvidia-smi

## 3. Install Ollama, cloudflared, gateway deps

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd ffmpeg > /dev/null
!which ollama || (curl -fsSL https://ollama.com/install.sh | sh)
!which cloudflared || (curl -fsSL -o /tmp/cf.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /tmp/cf.deb)
!pip install -q fastapi "uvicorn[standard]" httpx python-multipart easyocr soundfile onnx onnxruntime-gpu
# If IndicConformer's remote code asks for an extra package on first load, the /warmup output names it — pip install it and rerun section 7.
!ollama --version && cloudflared --version

## 4. Start Ollama (brain) on GPU0

In [ ]:
import shutil, subprocess, time, urllib.request
from pathlib import Path

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_BIN = shutil.which("ollama")
OLLAMA_MODELS = Path("/kaggle/working/ollama-models"); OLLAMA_MODELS.mkdir(parents=True, exist_ok=True)

OLLAMA_ENV = os.environ.copy()
OLLAMA_ENV.update({
    "CUDA_VISIBLE_DEVICES": OLLAMA_GPUS,
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_MODELS": str(OLLAMA_MODELS),
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
    "OLLAMA_CONTEXT_LENGTH": str(NUM_CTX),
    "OLLAMA_NUM_PARALLEL": str(NUM_PARALLEL),
    "OLLAMA_MAX_LOADED_MODELS": "1",
})

def http_ok(url):
    try:
        urllib.request.urlopen(url, timeout=2).close(); return True
    except Exception:
        return False

if not http_ok(f"{OLLAMA_URL}/api/tags"):
    ollama_proc = subprocess.Popen([OLLAMA_BIN, "serve"], env=OLLAMA_ENV,
                                   stdout=open("/tmp/ollama.log", "a"), stderr=subprocess.STDOUT,
                                   start_new_session=True)  # survives cell interrupts
    for _ in range(60):
        if http_ok(f"{OLLAMA_URL}/api/tags"): break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama not ready — see /tmp/ollama.log")
print("✅ Ollama ready on GPU", OLLAMA_GPUS)

## 5. Pull the brain model

In [ ]:
r = subprocess.run([OLLAMA_BIN, "pull", BRAIN_MODEL], env=OLLAMA_ENV)
if r.returncode != 0:
    print("Standard pull failed; retrying with --insecure (HF/Xet redirect workaround)...")
    subprocess.run([OLLAMA_BIN, "pull", "--insecure", BRAIN_MODEL], env=OLLAMA_ENV, check=True)
subprocess.run([OLLAMA_BIN, "list"], env=OLLAMA_ENV)

## 6. Warm up + check tool calling and Marathi

In [ ]:
import json

def ollama_chat(messages, tools=None):
    payload = {"model": BRAIN_MODEL, "messages": messages, "stream": False, "keep_alive": -1,
               "think": False, "options": {"num_ctx": NUM_CTX, "temperature": 0.2}}
    if tools: payload["tools"] = tools
    req = urllib.request.Request(f"{OLLAMA_URL}/api/chat", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=600) as resp:
        return json.loads(resp.read())["message"]

t = time.perf_counter()
print(ollama_chat([{"role": "user", "content": "एका वाक्यात सांगा: शिष्यवृत्ती अर्जासाठी उत्पन्न दाखला का लागतो?"}])["content"])
print(f"(first call incl. load: {time.perf_counter()-t:.1f}s)")

tools = [{"type": "function", "function": {
    "name": "get_required_documents", "description": "Look up required documents for a scheme",
    "parameters": {"type": "object", "properties": {"scheme": {"type": "string"}}, "required": ["scheme"]}}}]
msg = ollama_chat([{"role": "user", "content": "What documents do I need for the MahaDBT EBC scholarship?"}], tools)
print("Tool calls:", msg.get("tool_calls"))
!ollama ps

## 7. Write + start the gateway (ASR + OCR on GPU1, proxy to Ollama)

In [ ]:
%%writefile /kaggle/working/gateway.py
"""Aster GPU gateway: one authenticated endpoint in front of Ollama (LLM + vision), ASR and OCR."""
import asyncio
import io
import os
import subprocess
import tempfile

import httpx
import numpy as np
from fastapi import FastAPI, File, Form, Header, HTTPException, Request, UploadFile
from fastapi.responses import StreamingResponse
from PIL import Image
from starlette.background import BackgroundTask

TOKEN = os.environ["GATEWAY_TOKEN"]
OLLAMA = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
ENABLE_ASR = os.environ.get("ENABLE_ASR", "1") == "1"
ENABLE_OCR = os.environ.get("ENABLE_OCR", "1") == "1"
OCR_GPU = os.environ.get("OCR_GPU", "1") == "1"
HF_TOKEN = os.environ.get("HF_TOKEN") or None
ASR_MODEL_ID = os.environ.get("ASR_MODEL_ID", "ai4bharat/indic-conformer-600m-multilingual")
ASR_LANGS = {"hi", "mr", "gu", "bn", "ta", "te", "kn", "ml", "pa", "or", "ur"}  # no English: backend routes en to Sarvam/Bhashini

app = FastAPI(title="Aster GPU Gateway")
client = httpx.AsyncClient(timeout=httpx.Timeout(600.0, connect=10.0))
gpu_lock = asyncio.Lock()  # one ASR/OCR job on the GPU at a time
_asr = None
_ocr = None


def check(authorization):
    if authorization != f"Bearer {TOKEN}":
        raise HTTPException(status_code=401, detail="unauthorized")


def load_asr():
    global _asr
    if _asr is None:
        from transformers import AutoModel
        _asr = AutoModel.from_pretrained(ASR_MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
    return _asr


def load_ocr():
    global _ocr
    if _ocr is None:
        import easyocr
        _ocr = easyocr.Reader(["mr", "hi", "en"], gpu=OCR_GPU)
    return _ocr


def run_asr(data: bytes, lang: str) -> str:
    import torch
    import torchaudio
    model = load_asr()
    with tempfile.TemporaryDirectory() as d:
        src, dst = os.path.join(d, "in"), os.path.join(d, "out.wav")
        with open(src, "wb") as f:
            f.write(data)
        # Browser MediaRecorder sends webm/opus; normalise to 16 kHz mono wav
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ac", "1", "-ar", "16000", dst], check=True)
        wav, sr = torchaudio.load(dst)
    wav = torch.mean(wav, dim=0, keepdim=True)
    return model(wav, lang, "ctc")


def run_ocr(data: bytes):
    img = np.array(Image.open(io.BytesIO(data)).convert("RGB"))
    results = load_ocr().readtext(img)
    lines = []
    for i, (box, text, conf) in enumerate(results):
        lines.append({
            "id": f"L{i}",
            "text": text,
            "confidence": round(float(conf), 4),
            "bbox": [[float(x), float(y)] for x, y in box],
        })
    return {"width": int(img.shape[1]), "height": int(img.shape[0]), "lines": lines}


@app.get("/health")
async def health():
    try:
        r = await client.get(f"{OLLAMA}/api/ps", timeout=5)
        ollama = r.status_code == 200
    except Exception:
        ollama = False
    return {"ollama": ollama, "asr_enabled": ENABLE_ASR, "asr_loaded": _asr is not None,
            "ocr_enabled": ENABLE_OCR, "ocr_loaded": _ocr is not None}


@app.post("/warmup")
async def warmup(authorization: str | None = Header(None)):
    check(authorization)
    out = {}
    if ENABLE_OCR:
        try:
            await asyncio.to_thread(load_ocr)
            out["ocr"] = "ok"
        except Exception as e:
            out["ocr"] = f"failed: {e}"
    if ENABLE_ASR:
        try:
            await asyncio.to_thread(load_asr)
            out["asr"] = "ok"
        except Exception as e:
            out["asr"] = f"failed: {e}"
    return out


@app.post("/asr")
async def asr(file: UploadFile = File(...), lang: str = Form("hi"), authorization: str | None = Header(None)):
    check(authorization)
    if not ENABLE_ASR:
        raise HTTPException(503, "asr disabled on this worker")
    if lang not in ASR_LANGS:
        raise HTTPException(400, f"lang '{lang}' not served locally; use the API fallback")
    data = await file.read()
    try:
        async with gpu_lock:
            text = await asyncio.to_thread(run_asr, data, lang)
    except Exception as e:
        raise HTTPException(503, f"asr failed: {e}")
    return {"text": text, "lang": lang, "engine": ASR_MODEL_ID}


@app.post("/ocr")
async def ocr(file: UploadFile = File(...), authorization: str | None = Header(None)):
    check(authorization)
    if not ENABLE_OCR:
        raise HTTPException(503, "ocr disabled on this worker")
    data = await file.read()
    try:
        async with gpu_lock:
            result = await asyncio.to_thread(run_ocr, data)
    except Exception as e:
        raise HTTPException(503, f"ocr failed: {e}")
    return {**result, "engine": "easyocr"}


@app.api_route("/v1/{path:path}", methods=["GET", "POST"])
async def openai_proxy(path: str, request: Request, authorization: str | None = Header(None)):
    """OpenAI-compatible passthrough to Ollama (chat, tools, vision, streaming)."""
    check(authorization)
    body = await request.body()
    req = client.build_request(
        request.method, f"{OLLAMA}/v1/{path}", content=body,
        headers={"Content-Type": request.headers.get("content-type", "application/json")},
    )
    resp = await client.send(req, stream=True)
    return StreamingResponse(
        resp.aiter_raw(), status_code=resp.status_code,
        media_type=resp.headers.get("content-type"), background=BackgroundTask(resp.aclose),
    )

In [ ]:
GATEWAY_URL = f"http://127.0.0.1:{GATEWAY_PORT}"
gw_env = os.environ.copy()
gw_env.update({
    "CUDA_VISIBLE_DEVICES": GATEWAY_GPUS,
    "GATEWAY_TOKEN": GATEWAY_TOKEN,
    "HF_TOKEN": HF_TOKEN or "",
    "OLLAMA_URL": OLLAMA_URL,
    "ENABLE_ASR": "1" if MODE == "fast" else "0",
    "ENABLE_OCR": "1",
    "OCR_GPU": "1" if MODE == "fast" else "0",
})
if not http_ok(f"{GATEWAY_URL}/health"):
    gateway_proc = subprocess.Popen(
        ["python", "-m", "uvicorn", "gateway:app", "--host", "127.0.0.1", "--port", str(GATEWAY_PORT)],
        cwd="/kaggle/working", env=gw_env, stdout=open("/tmp/gateway.log", "a"), stderr=subprocess.STDOUT,
        start_new_session=True)  # survives cell interrupts
    for _ in range(60):
        if http_ok(f"{GATEWAY_URL}/health"): break
        time.sleep(1)
    else:
        raise RuntimeError("Gateway not ready — see /tmp/gateway.log")

# Load ASR + OCR weights now so the first user request isn't slow
req = urllib.request.Request(f"{GATEWAY_URL}/warmup", method="POST",
                             headers={"Authorization": f"Bearer {GATEWAY_TOKEN}"})
with urllib.request.urlopen(req, timeout=900) as resp:
    print("Warmup:", json.loads(resp.read()))
print(urllib.request.urlopen(f"{GATEWAY_URL}/health").read().decode())
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

## 8. Cloudflare Quick Tunnel → gateway (not raw Ollama)

In [ ]:
import queue, re, threading

def _pipe(proc, q):
    for line in iter(proc.stdout.readline, ""):
        q.put(line)

def start_tunnel():
    """(Re)start the quick tunnel to the gateway; returns the new public URL."""
    global cf_proc, PUBLIC_URL
    try: cf_proc.kill()
    except Exception: pass
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2", "--edge-ip-version", "4",
         "--url", GATEWAY_URL],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        start_new_session=True)  # survives cell interrupts
    q = queue.Queue(); threading.Thread(target=_pipe, args=(cf_proc, q), daemon=True).start()
    PUBLIC_URL, deadline = None, time.monotonic() + 60
    while time.monotonic() < deadline:
        try: line = q.get(timeout=1)
        except queue.Empty: continue
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m: PUBLIC_URL = m.group(0); break
    if not PUBLIC_URL:
        raise RuntimeError("No tunnel URL — rerun this cell")
    time.sleep(10)  # let Cloudflare route the new hostname
    return PUBLIC_URL

if not ("cf_proc" in globals() and cf_proc.poll() is None and globals().get("PUBLIC_URL")):
    start_tunnel()
print("Public gateway:", PUBLIC_URL)
print("OpenAI base URL:", f"{PUBLIC_URL}/v1", "| model:", BRAIN_MODEL)

## 9. Register the URL in Supabase (backend auto-discovers it)

In [ ]:
SUPABASE_URL = (SUPABASE_URL or "").strip().rstrip("/")
SUPABASE_ANON_KEY = (SUPABASE_ANON_KEY or "").strip()
GPU_REGISTER_SECRET = (GPU_REGISTER_SECRET or "").strip()

def register():
    if not (SUPABASE_URL and SUPABASE_ANON_KEY and GPU_REGISTER_SECRET):
        print("Supabase secrets not set — copy PUBLIC_URL into the backend env manually."); return False
    body = {"p_secret": GPU_REGISTER_SECRET, "p_name": WORKER_NAME, "p_url": PUBLIC_URL,
            "p_models": {"brain": BRAIN_MODEL, "mode": MODE, "asr": MODE == "fast", "ocr": True}}
    req = urllib.request.Request(f"{SUPABASE_URL}/rest/v1/rpc/register_gpu", data=json.dumps(body).encode(),
        headers={"apikey": SUPABASE_ANON_KEY, "Content-Type": "application/json"}, method="POST")
    try:
        urllib.request.urlopen(req, timeout=15).close(); return True
    except urllib.error.HTTPError as e:
        print("Register failed:", e.code, e.read().decode()); return False
    except Exception as e:
        print("Register failed:", e); return False

print("Secret check -> length:", len(GPU_REGISTER_SECRET), "starts with:", GPU_REGISTER_SECRET[:4])
print("Registered:", register())

## 10. End-to-end test through the tunnel (exactly what the backend does)

In [ ]:
def via_tunnel(path, payload):
    req = urllib.request.Request(f"{PUBLIC_URL}{path}", data=json.dumps(payload).encode(), method="POST",
        headers={"Authorization": f"Bearer {GATEWAY_TOKEN}", "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.loads(resp.read())

t = time.perf_counter()
out = via_tunnel("/v1/chat/completions", {"model": BRAIN_MODEL, "messages": [
    {"role": "system", "content": "You are Aster, a friendly form-filling assistant. Reply in the user's language."},
    {"role": "user", "content": "मला MahaDBT फॉर्म भरायचा आहे, सुरुवात कशी करू?"}], "max_tokens": 200})
print(out["choices"][0]["message"]["content"])
print(f"Round trip via tunnel: {time.perf_counter()-t:.1f}s")

# Unauthenticated request must be rejected
try:
    urllib.request.urlopen(urllib.request.Request(f"{PUBLIC_URL}/v1/models"), timeout=15)
    print("⚠️ Gateway accepted a request without token!")
except urllib.error.HTTPError as e:
    print("✅ No-token request rejected:", e.code)

## 11. Keep alive + heartbeat (leave running; required for Save & Run All)

In [ ]:
# Safe to stop with ■ — background processes run in their own session now.
def tunnel_ok():
    try:
        urllib.request.urlopen(f"{PUBLIC_URL}/health", timeout=10).close(); return True
    except Exception:
        return False

fails = 0
while True:
    ok_gw, ok_llm = http_ok(f"{GATEWAY_URL}/health"), http_ok(f"{OLLAMA_URL}/api/tags")
    ok_tunnel = cf_proc.poll() is None and tunnel_ok()
    fails = 0 if ok_tunnel else fails + 1
    if fails >= 3 or cf_proc.poll() is not None:
        start_tunnel(); fails = 0
        print("🔁 New public URL:", PUBLIC_URL, flush=True)
        ok_tunnel = tunnel_ok()
    if ok_tunnel and ok_gw and ok_llm:
        register()
    print(time.strftime("%H:%M:%S"), "| tunnel", ok_tunnel, "| gateway", ok_gw, "| ollama", ok_llm,
          "|", PUBLIC_URL, flush=True)
    time.sleep(60)

## 12. Status check (run anytime; stop cell 11 first)

In [ ]:
print("tunnel process alive:", cf_proc.poll() is None)
print("gateway:", http_ok(f"{GATEWAY_URL}/health"), "| ollama:", http_ok(f"{OLLAMA_URL}/api/tags"))
print("public URL:", PUBLIC_URL)
!tail -n 5 /tmp/ollama.log; tail -n 5 /tmp/gateway.log